In [2]:
import sys, pandas as pd
from collections import Counter
sys.path.append("..")

from src.parser import split_ingredients, parse_ingredient

df = pd.read_csv("../data/processed/food_safety_clean (1).csv")
df["items"] = df["ingredients_clean"].fillna("").map(split_ingredients)
df["canon3"] = df["items"].map(lambda L: [c for x in L for c in parse_ingredient(x)])
print(df.shape, "| example:", df["canon3"].iloc[0][:6])

(1526, 71) | example: ['nougat caramel sugar', 'glucose syrup', 'hydrogenated vegetable fats', 'malt extract', 'milk solids', 'emulsifier']


In [3]:
nut = ["sugars_g", "salt_g", "saturated_fat_g", "energy_kcal"]
ex = df.explode("canon3")
hint = (ex.groupby("canon3")[nut].median().round(1)
          .join(ex["canon3"].value_counts().rename("n")))
hint = hint[hint["n"] >= 15].sort_values("n", ascending=False).head(100)
hint.to_csv("../data/processed/ingredient_nutrient_hints.csv")
hint.head(30)

,sugars_g,salt_g,saturated_fat_g,energy_kcal,n
canon3,,,,,
salt,4.8,1.1,5.1,382.0,654
sugar,16.4,0.6,4.2,376.0,587
added flavour,15.5,0.5,5.5,417.0,559
emulsifier,24.3,0.5,9.0,445.0,378
acidity regulator,7.4,1.2,2.1,350.0,350
milk solids,21.5,0.4,8.3,413.0,265
water,4.2,0.6,0.7,119.0,234
colour,13.7,0.5,2.5,329.6,223
ins 330,7.8,1.2,4.6,390.0,191


In [7]:
kb = pd.read_csv("../data/knowledgebase/ingredient.csv")
known = set(kb["ingredient"])
mentions = [x for L in df["canon3"] for x in L]
hit = sum(1 for m in mentions if m in known)
print(f"KB recognizes {hit/len(mentions):.1%} of ingredient mentions")
for name, n in Counter(m for m in mentions if m not in known).most_common(30):
    print(n, name)

KB recognizes 34.7% of ingredient mentions
191 ins 330
145 ins 322
143 ins 500
126 ins 471
86 flavour enhancer
83 black pepper
69 anticaking agent
66 ins 503
58 ins 415
58 ins 150d
57 ins 412
56 ins 260
55 wheat
54 ins 551
48 ins 300
47 ins 627
46 turmeric
44 ins 631
44 cocoa powder
43 garlic powder
40 ins 472e
40 edible vegetable oil palm oil
40 turmeric powder
39 ins 407
38 cinnamon
38 soy
38 ins 202
37 vitamins
37 black salt
37 ins 110


In [8]:
a = pd.read_csv("../data/knowledgebase/ingredient.csv")
b = pd.read_csv("../data/knowledgebase/ingredients_batch2.csv")
kb = pd.concat([a, b]).drop_duplicates("ingredient", keep="last")
kb.to_csv("../data/knowledgebase/ingredient.csv", index=False)
print(len(kb), "rows")

70 rows


In [9]:
from src.kb import load_kb, lookup
kb = load_kb()
how = Counter(lookup(m, kb)[1].split(":")[0] for m in mentions)
total = sum(how.values())
print({k: f"{v/total:.1%}" for k, v in how.items()})

{'unrecognized': '54.6%', 'exact': '43.1%', 'fallback': '2.3%'}
